# Cattle Genomic Selection, Breeding Methods, Preventive Health, and IVF-MOET Workflow Records Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.fd0p-f11x/fair2.json`

### Notes
- This notebook uses the Croissant metadata as the entry point for exploration.
- All dataset entities should be referenced by their `@id` where available.
- This dataset appears to expose rich dataset-level metadata. If record sets are available through the Croissant endpoint, the notebook will discover and load them dynamically.
- If no record sets are declared, the notebook still provides a metadata-first exploration workflow.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata from the Croissant URL and inspect the dataset object.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.fd0p-f11x/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Convert metadata to a JSON-like object for easier inspection when needed
metadata_json = metadata.to_json()

print('Dataset loaded successfully.')
print('Name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))
print('Identifier:', metadata_json.get('identifier', 'N/A'))
print('Version:', metadata_json.get('version', 'N/A'))
print('License:', metadata_json.get('license', 'N/A'))

## 2. Dataset Metadata Overview
Review the top-level metadata fields and summarize the dataset context before attempting record extraction.

In [ ]:
# Display a compact metadata summary
summary_fields = {
    '@id': metadata_json.get('@id'),
    '@type': metadata_json.get('@type'),
    'name': metadata_json.get('name'),
    'datePublished': metadata_json.get('datePublished'),
    'spatialCoverage': metadata_json.get('spatialCoverage'),
    'temporalCoverage': metadata_json.get('temporalCoverage'),
    'citeAs': metadata_json.get('citeAs'),
    'conformsTo': metadata_json.get('conformsTo')
}

pd.DataFrame(summary_fields.items(), columns=['field', 'value'])

In [ ]:
# Show selected descriptive lists from the metadata
list_fields = [
    'keywords',
    'dataUseCases',
    'dataLimitations',
    'dataBiases',
    'personalSensitiveInformation'
]

for field_name in list_fields:
    value = metadata_json.get(field_name, [])
    print(f'\n{field_name}:')
    if isinstance(value, list):
        for i, item in enumerate(value, start=1):
            print(f'  {i}. {item}')
    else:
        print(' ', value)

## 3. Data Overview
Discover available record sets and inspect their `@id` values.

The notebook uses variables so that downstream code adapts automatically to the available Croissant structure.

In [ ]:
# Discover record sets from the metadata JSON
record_sets_meta = metadata_json.get('recordSet', []) or []

print(f'Number of record sets declared in metadata: {len(record_sets_meta)}')

record_set_rows = []
for rs in record_sets_meta:
    record_set_rows.append({
        'record_set_id': rs.get('@id'),
        'name': rs.get('name'),
        'description': rs.get('description'),
        'field_count': len(rs.get('field', []) or [])
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df

In [ ]:
# Build a mapping from record set @id to its fields
record_set_field_map = {}

for rs in record_sets_meta:
    rs_id = rs.get('@id')
    fields = rs.get('field', []) or []
    field_rows = []
    for field in fields:
        field_rows.append({
            'field_id': field.get('@id'),
            'field_name': field.get('name'),
            'description': field.get('description'),
            'dataType': field.get('dataType'),
            'source': field.get('source'),
            'references_column_id': (field.get('source') or {}).get('column') if isinstance(field.get('source'), dict) else None,
            'references_extract': (field.get('source') or {}).get('extract') if isinstance(field.get('source'), dict) else None
        })
    record_set_field_map[rs_id] = pd.DataFrame(field_rows)

if record_set_field_map:
    first_record_set_id = next(iter(record_set_field_map.keys()))
    print('Example record set @id:', first_record_set_id)
    display(record_set_field_map[first_record_set_id].head(20))
else:
    print('No record sets were declared in the Croissant metadata.')

## 4. Record Inspection
If record sets are available, inspect example rows using the record set `@id`.

This section is written defensively so the notebook remains usable even when the dataset provides metadata without materialized record sets.

In [ ]:
record_set_ids = [rs.get('@id') for rs in record_sets_meta if rs.get('@id')]
print('Discovered record set @ids:')
for rs_id in record_set_ids:
    print('-', rs_id)

if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print(f'\nInspecting up to 5 records from record set: {sample_record_set_id}')
    try:
        for i, x in enumerate(dataset.records(record_set=sample_record_set_id)):
            print(x)
            if i >= 4:
                break
    except Exception as e:
        print('Could not iterate over records for this record set.')
        print('Reason:', e)
else:
    print('No record set @ids available for record inspection.')

## 5. Data Extraction
Load records from each available record set into pandas DataFrames using record set `@id` values.

When records are available, this creates a dictionary keyed by record set `@id`.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'{record_set_id}: loaded {len(dataframes[record_set_id])} records')
    except Exception as e:
        print(f'{record_set_id}: failed to load records')
        print('Reason:', e)

print('\nLoaded DataFrames for record sets:')
print(list(dataframes.keys()))

In [ ]:
# Inspect columns for the first successfully loaded record set
loaded_record_set_ids = list(dataframes.keys())

if loaded_record_set_ids:
    example_record_set_id = loaded_record_set_ids[0]
    print('Example record set @id:', example_record_set_id)
    print('Columns:')
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No record sets were loaded into DataFrames.')

## 6. Exploratory Data Analysis (EDA)
Apply basic data processing to one loaded record set.

Because Croissant datasets vary in schema, the notebook selects fields dynamically:
- a record set `@id`
- a candidate numeric column if one exists
- a candidate grouping column if one exists

If no numeric field is available, the notebook falls back to categorical summaries.

In [ ]:
eda_record_set_id = loaded_record_set_ids[0] if loaded_record_set_ids else None

if eda_record_set_id is not None:
    eda_df = dataframes[eda_record_set_id].copy()
    print(f'Using record set for EDA: {eda_record_set_id}')
    print('Shape:', eda_df.shape)
    display(eda_df.head())
else:
    eda_df = pd.DataFrame()
    print('No loaded record set available for EDA.')

In [ ]:
# Detect numeric and categorical columns
if not eda_df.empty:
    numeric_columns = eda_df.select_dtypes(include=['number']).columns.tolist()
    categorical_columns = eda_df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

    print('Numeric columns:', numeric_columns)
    print('Categorical/text columns:', categorical_columns)
else:
    numeric_columns = []
    categorical_columns = []

In [ ]:
# Example processing for a numeric field, if available
if not eda_df.empty and numeric_columns:
    numeric_field_id = numeric_columns[0]
    series = eda_df[numeric_field_id].dropna()

    threshold = series.median()
    filtered_df = eda_df[eda_df[numeric_field_id] > threshold].copy()

    print(f'Selected numeric field: {numeric_field_id}')
    print(f'Filtering records where {numeric_field_id} > median threshold ({threshold})')
    display(filtered_df.head())

    std_value = filtered_df[numeric_field_id].std()
    if std_value and pd.notna(std_value) and std_value != 0:
        filtered_df[f'{numeric_field_id}_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / std_value
        print(f'Normalized field created: {numeric_field_id}_normalized')
        display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())
    else:
        print('Normalization skipped because standard deviation is zero or undefined.')

    group_field_id = categorical_columns[0] if categorical_columns else None
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f'Grouped mean of {numeric_field_id} by {group_field_id}:')
        display(grouped_df.head())
    else:
        print('No categorical grouping field available for grouped analysis.')
else:
    print('No numeric columns available; skipping numeric EDA.')

In [ ]:
# Fallback categorical/text exploration
if not eda_df.empty and categorical_columns:
    category_field_id = categorical_columns[0]
    print(f'Selected categorical/text field: {category_field_id}')
    value_counts_df = eda_df[category_field_id].astype(str).value_counts(dropna=False).reset_index()
    value_counts_df.columns = [category_field_id, 'count']
    display(value_counts_df.head(10))
else:
    print('No categorical/text columns available for fallback exploration.')

## 7. Visualization
Visualize distributions or grouped summaries when record data is available.

The plotting logic adapts automatically to the detected schema.

In [ ]:
if not eda_df.empty and numeric_columns:
    numeric_field_id = numeric_columns[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(eda_df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if categorical_columns:
        group_field_id = categorical_columns[0]
        plot_df = eda_df[[group_field_id, numeric_field_id]].dropna().copy()
        top_groups = plot_df[group_field_id].astype(str).value_counts().head(10).index
        plot_df = plot_df[plot_df[group_field_id].astype(str).isin(top_groups)]

        if not plot_df.empty:
            plt.figure(figsize=(10, 5))
            sns.boxplot(data=plot_df, x=group_field_id, y=numeric_field_id)
            plt.title(f'{numeric_field_id} by {group_field_id}')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
else:
    print('Numeric visualizations skipped because no numeric columns were detected.')

if not eda_df.empty and not numeric_columns and categorical_columns:
    category_field_id = categorical_columns[0]
    plot_counts = eda_df[category_field_id].astype(str).value_counts().head(10)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=plot_counts.index, y=plot_counts.values)
    plt.title(f'Top values in {category_field_id}')
    plt.xlabel(category_field_id)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 8. Metadata-Centric Exploration of Distributions and References
Even if record sets are absent, the dataset metadata contains useful linked entities such as distributions and authors. This section inspects those references using their `@id` values.

In [ ]:
# Explore linked distributions by @id
distribution_items = metadata_json.get('distribution', []) or []
distribution_df = pd.DataFrame([
    {'distribution_id': item.get('@id')}
    for item in distribution_items
])

print(f'Number of distributions: {len(distribution_df)}')
display(distribution_df)

In [ ]:
# Explore author references by @id
author_items = metadata_json.get('author', []) or []
author_df = pd.DataFrame([
    {'author_id': item.get('@id')}
    for item in author_items
])

print(f'Number of authors: {len(author_df)}')
display(author_df)

## 9. Conclusion
This notebook demonstrated how to:

- Load a Croissant dataset with `mlcroissant`
- Access dataset metadata through `dataset.metadata` as a single object
- Discover record sets and fields dynamically using their `@id` values
- Load records into pandas DataFrames when record sets are available
- Perform adaptive exploratory analysis and visualization
- Fall back to metadata-centric exploration when tabular record sets are not exposed

For this dataset, the metadata indicates a domain-specific resource focused on cattle breeding, genomic selection, preventive health, and IVF/MOET workflow documentation. If future versions of the Croissant file expose record sets, the same notebook structure can be reused directly without major changes.